# Compute metrics for different runs and plot them
##### author: Elizabeth A. Barnes, Randal J. Barnes and Mark DeMaria
##### version: v0.2.0

In [1]:
import sys
import os

sys.path.append("..")

import numpy as np
import pandas as pd
import pprint

import silence_tensorflow.auto
import tensorflow as tf

import warnings
warnings.filterwarnings("ignore")

import experiment_settings
import build_model
from build_data import build_hurricane_data

In [2]:
__author__ = "Randal J Barnes and Elizabeth A. Barnes"
__version__ = "08 October 2022"

silence_tensorflow()
tf.config.set_visible_devices([], "GPU")  # turn-off tensorflow-metal if it is on

DATA_PATH = "../data/"
MODEL_PATH = "saved_models/"
FIGURE_PATH = "figures/analysis_plots/"
PREDICTION_PATH = "saved_predictions/"

OVERWRITE_METRICS = False

In [3]:
mpl.rcParams["figure.facecolor"] = "white"
dpiFig = 300.0
np.warnings.filterwarnings("ignore", category=np.VisibleDeprecationWarning)

In [4]:
EXP_NAME_VEC = ("centered_bivariate_normal_201_AL24",
                "centered_bivariate_normal_202_AL48",
                "centered_bivariate_normal_203_AL72",
                "centered_bivariate_normal_204_AL96",
                "centered_bivariate_normal_205_AL120",

                "centered_bivariate_normal_101_EPCP24",
                "centered_bivariate_normal_102_EPCP48",
                "centered_bivariate_normal_103_EPCP72",
                "centered_bivariate_normal_104_EPCP96",
                "centered_bivariate_normal_105_EPCP120",
                )

## Compute Predictions for ALL of the data

In [5]:
for exp_name in EXP_NAME_VEC:
    settings = experiment_settings.get_settings(exp_name)
    print(exp_name)

    # set testing data
    if settings["test_condition"] == "leave-one-out":
        TESTING_YEARS_LIST = np.arange(2013,2022)
    elif settings["test_condition"] == "years":
        TESTING_YEARS_LIST = (np.copy(settings["years_test"]))
    else:
        raise NotImplementedError('no such testing condition')

    for testing_years in TESTING_YEARS_LIST:
        # set testing year
        settings["years_test"] = (testing_years,)


        for rng_seed in settings['rng_seed_list']:
            settings['rng_seed'] = rng_seed
            NETWORK_SEED_LIST = [settings["rng_seed"]]
            network_seed = NETWORK_SEED_LIST[0]
            tf.random.set_seed(network_seed)  # This sets the global random seed.

            model_name = (
                exp_name + "_" +
                str(testing_years) + '_' +
                settings["uncertainty_type"] + '_' +
                f"network_seed_{network_seed}_rng_seed_{settings['rng_seed']}"
            )
            #----------------------------------------------------------------------------------------------------
            # check if the metric filename exists already
            metric_filename = PREDICTION_PATH + model_name + '_allPredictions.csv'
            if (os.path.exists(metric_filename) and OVERWRITE_METRICS==False):
                print(metric_filename + ' exists. Skipping...')
                continue
            #----------------------------------------------------------------------------------------------------
            # get the data
            (
                data_summary,
                x_train,
                onehot_train,
                x_val,
                onehot_val,
                x_test,
                onehot_test,
                x_valtest,
                onehot_valtest,
                df_train,
                df_val,
                df_test,
                df_valtest,
            ) = build_hurricane_data(DATA_PATH, settings, verbose=0)

            #----------------------------------------------------------------------------------------------------
            # get the model
            # Make, compile, and train the model
            tf.keras.backend.clear_session()
            model = build_model.make_model(
                settings,
                x_train,
                onehot_train,
                model_compile=False,
            )
#----------------------------------------------------------------------------------------------------
            # load the model
            try:
                model.load_weights(MODEL_PATH + model_name + "_weights.h5")
            except:
                print(model_name + ': model does not exist. skipping...')
                continue

            # get metrics and put into a dictionary
            pprint.pprint(model_name)

            # concatenate all input data together in a consistent order
            x_data = np.concatenate([x_train, x_val])
            x_data = np.concatenate([x_data, x_test])
            # print(np.shape(x_data), len(x_train)+len(x_val)+len(x_test))

            # concatenate all output data together in a consistent order
            onehot_data = np.concatenate([onehot_train, onehot_val])
            onehot_data = np.concatenate([onehot_data, onehot_test])
            # print(np.shape(onehot_data), len(onehot_train)+len(onehot_val)+len(onehot_test))

            # concatenate all dataframes together in a consistent order
            df_data = pd.concat([df_train, df_val])
            df_data = pd.concat([df_data, df_test])
            # print(np.shape(df_data), len(df_train)+len(df_val)+len(df_test))

            # get prediction metrics of interest
            predictions = model.predict(x_data)

            # get and add predictions to the data_frame
            df_predictions = df_data.copy()
            df_predictions["mu_u"] = predictions[:,0]
            df_predictions["mu_v"] = predictions[:,1]
            df_predictions["sigma_u"] = predictions[:,2]
            df_predictions["sigma_v"] = predictions[:,3]
            df_predictions["rho"] = predictions[:,4]

            # save the dataframe
            df_predictions.to_csv(metric_filename)


centered_bivariate_normal_201_AL24
saved_predictions/centered_bivariate_normal_201_AL24_2013_centered_bivariate_normal_network_seed_123_rng_seed_123_allPredictions.csv exists. Skipping...
saved_predictions/centered_bivariate_normal_201_AL24_2013_centered_bivariate_normal_network_seed_234_rng_seed_234_allPredictions.csv exists. Skipping...
saved_predictions/centered_bivariate_normal_201_AL24_2013_centered_bivariate_normal_network_seed_345_rng_seed_345_allPredictions.csv exists. Skipping...
saved_predictions/centered_bivariate_normal_201_AL24_2014_centered_bivariate_normal_network_seed_123_rng_seed_123_allPredictions.csv exists. Skipping...
saved_predictions/centered_bivariate_normal_201_AL24_2014_centered_bivariate_normal_network_seed_234_rng_seed_234_allPredictions.csv exists. Skipping...
saved_predictions/centered_bivariate_normal_201_AL24_2014_centered_bivariate_normal_network_seed_345_rng_seed_345_allPredictions.csv exists. Skipping...
saved_predictions/centered_bivariate_normal_201